In [1]:
!curl -o leads.csv https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  272k  100  272k    0     0   333k      0 --:--:-- --:--:-- --:--:--  333k


In [3]:
import pandas as pd
import numpy as np
df=pd.read_csv('leads.csv')
df.shape
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [17]:

df['industry'].mode()[0]

'technology'

In [18]:
num_list=['interaction_count','lead_score','number_of_courses_viewed','annual_income']
df[num_list].corr()

,interaction_count,lead_score,number_of_courses_viewed,annual_income
interaction_count,1.000000,0.915746,0.721609,0.122842
lead_score,0.915746,1.000000,0.757204,0.229496
number_of_courses_viewed,0.721609,0.757204,1.000000,0.161300
annual_income,0.122842,0.229496,0.161300,1.000000


In [19]:
df.isnull().sum()
for c in df.select_dtypes(include='object').columns:
    df[c]=df[c].fillna('unknown')
for c in df.select_dtypes(include='number').columns:
    df[c]=df[c].fillna(0)

In [20]:
from sklearn.metrics import mutual_info_score
categorical=['lead_source','industry','employment_status','location']
for c in categorical:
    mi=mutual_info_score(df[c],df['lead_score'])
print(f'{c}:{mi:.4f}')

location:0.0511


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/cluster/_supervised.py:58: UserWarning: Clustering metrics expects discrete values but received multiclass values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/cluster/_supervised.py:58: UserWarning: Clustering metrics expects discrete values but received multiclass values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/cluster/_supervised.py:58: UserWarning: Clustering metrics expects discrete values but received multiclass values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/cluster/_supervised.py:58: UserWarning: Clustering metrics expects discrete values but received multiclass values for label, and continuous values for target
  warnings.warn(msg, UserW

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
df['lead_score']=(df['lead_score']>0.5).astype(int)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1)
df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)
y_train=df_train['lead_score'].values
y_val=df_val['lead_score'].values
y_test=df_test['lead_score'].values
del df_train['lead_score']
del df_val['lead_score']
del df_test['lead_score']

dv=DictVectorizer(sparse=False)
train_dict=df_train.to_dict(orient='records')
X_train=dv.fit_transform(train_dict)
val_dict=df_val.to_dict(orient='records')
X_val=dv.transform(val_dict)
model=LogisticRegression(solver='lbfgs',max_iter=5000)
model.fit(X_train,y_train)
y_pread=model.predict_proba(X_val)[:,1]
churn_decision=(y_pread>=0.5)
accuracy=((y_pread>=0.5)==y_val).mean()
print(f'Accuracy:{accuracy:2f}')

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Accuracy:0.913000


In [40]:
features=['lead_source','number_of_courses_viewed','interaction_count']
for f in features:
    df_train_f=df_train.drop(columns=[f])
    df_val_f=df_val.drop(columns=[f])
    dv_f=DictVectorizer(sparse=False)
    X_train_f=dv_f.fit_transform(df_train_f.to_dict(orient='records'))
    X_val_f=dv_f.fit_transform(df_val_f.to_dict(orient='records'))
    model_f=LogisticRegression(solver='lbfgs',max_iter=5000)
    model.fit(X_train_f,y_train)
    y_pread_f=model.predict_proba(X_val_f)[:,1]
    accuracy=((y_pread_f>=0.5)==y_val).mean()
    print(f'after droppin{f}:Accuracy={accuracy:.4f}')

after droppinlead_source:Accuracy=0.8990
after droppinnumber_of_courses_viewed:Accuracy=0.9110
after droppininteraction_count:Accuracy=0.8190


In [47]:
dv=DictVectorizer(sparse=False)
train_dict=df_train.to_dict(orient='records')
X_train=dv.fit_transform(train_dict)
val_dict=df_val.to_dict(orient='records')
X_val=dv.transform(val_dict)
for C in[0.000001,0.00001,0.0001,0.001]:
    model_c=LogisticRegression(solver='lbfgs',C=C,max_iter=5000)
    model_c.fit(X_train,y_train)
    y_pread_c=model_c.predict_proba(X_val)[:,1]
    accuracy=((y_pread_c>=0.5)==y_val).mean()
    print(f'c={C},accuracy={accuracy:.4f}')

c=1e-06,accuracy=0.6180
c=1e-05,accuracy=0.6650
c=0.0001,accuracy=0.8480
c=0.001,accuracy=0.9020
